In [1]:
from datetime import datetime
import logging
import os
import sys
from typing import Any, Optional
from dotenv import find_dotenv, load_dotenv
from pathlib import Path

# Add the parent directory to the Python path to import the sample_helper module
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'python'))
sys.path.insert(0, str(Path().resolve()))
sys.path.insert(0, str(Path("notebooks/02-azure-content-understanding").resolve()))

from dotenv import load_dotenv
load_dotenv() 

from content_understanding_client import AzureContentUnderstandingClient
from sample_helper import save_json_to_file 
from azure.identity import DefaultAzureCredential

load_dotenv(find_dotenv())
logging.basicConfig(level=logging.INFO)

# For authentication, you can use either token-based auth or subscription key; only one is required
AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT")
# IMPORTANT: Replace with your actual subscription key or set it in your ".env" file if not using token authentication
AZURE_AI_API_KEY = os.getenv("AZURE_AI_API_KEY")
API_VERSION = "2025-11-01"

# Create token provider for Azure AD authentication
def token_provider():
    credential = DefaultAzureCredential()
    token = credential.get_token("https://cognitiveservices.azure.com/.default")
    return token.token

# Create the Content Understanding client
try:
    client = AzureContentUnderstandingClient(
        endpoint=AZURE_AI_ENDPOINT,
        api_version=API_VERSION,
        subscription_key=AZURE_AI_API_KEY,
        token_provider=token_provider if not AZURE_AI_API_KEY else None,
        x_ms_useragent="azure-ai-content-understanding-python-sample-ga"    # The user agent is used for tracking sample usage and does not provide identity information. You can change this if you want to opt out of tracking.
    )
    credential_type = "Subscription Key" if AZURE_AI_API_KEY else "Azure AD Token"
    print(f"✅ Client created successfully")
    print(f"   Endpoint: {AZURE_AI_ENDPOINT}")
    print(f"   Credential: {credential_type}")
    print(f"   API Version: {API_VERSION}")
except Exception as e:
    credential_type = "Subscription Key" if AZURE_AI_API_KEY else "Azure AD Token"
    print(f"❌ Failed to create client")
    print(f"   Endpoint: {AZURE_AI_ENDPOINT}")
    print(f"   Credential: {credential_type}")
    print(f"   Error: {e}")
    raise

✅ Client created successfully
   Endpoint: https://azure-foundry-westus-resource.services.ai.azure.com/
   Credential: Subscription Key
   API Version: 2025-11-01


In [2]:
import os
import json
from datetime import datetime


def process_and_save_analysis(
    analysis_result: dict,
    output_prefix: str = "analysis",
    output_dir: str = "test_output",
    print_fields: bool = True,
    save_json: bool = True,
):
    """
    Processes Azure Content Understanding analysis result:
      - Prints extracted fields
      - Prints content metadata
      - Saves full JSON to file (timestamped)

    Args:
        analysis_result (dict): Full response from client.poll_result(...)
        output_prefix (str): Prefix for saved JSON file (e.g., 'raw', 'normalized')
        output_dir (str): Directory to save output files
        print_fields (bool): Whether to print extracted fields
        save_json (bool): Whether to save full JSON to disk
    """

    if not analysis_result or "result" not in analysis_result:
        print("❌ No valid analysis result available.")
        return None

    result = analysis_result["result"]
    contents = result.get("contents", [])

    if not contents:
        print("⚠️ No contents found in result.")
        return None

    first_content = contents[0]
    fields = first_content.get("fields", {})

    # -----------------------------
    # 1️⃣ Print extracted fields
    # -----------------------------
    if print_fields:
        print("\n📊 Extracted Fields:")
        print("-" * 80)

        if fields:
            for field_name, field_value in fields.items():
                field_type = field_value.get("type")

                if field_type == "string":
                    print(f"{field_name}: {field_value.get('valueString')}")

                elif field_type == "number":
                    print(f"{field_name}: {field_value.get('valueNumber')}")

                elif field_type == "array":
                    items = field_value.get("valueArray", [])
                    print(f"{field_name} (array with {len(items)} items):")

                    for idx, item in enumerate(items, 1):
                        if item.get("type") == "object":
                            print(f"  Item {idx}:")
                            for key, val in item.get("valueObject", {}).items():
                                if val.get("type") == "string":
                                    print(f"    {key}: {val.get('valueString')}")
                                elif val.get("type") == "number":
                                    print(f"    {key}: {val.get('valueNumber')}")

                elif field_type == "object":
                    print(f"{field_name}: {field_value.get('valueObject')}")

                print()
        else:
            print("No fields extracted")

        # -----------------------------
        # 2️⃣ Print metadata
        # -----------------------------
        print("\n📋 Content Metadata:")
        print("-" * 80)
        print(f"Kind: {first_content.get('kind')}")
        print(f"Pages: {first_content.get('startPageNumber')} - {first_content.get('endPageNumber')}")
        print(f"Unit: {first_content.get('unit')}")
        print()

    # -----------------------------
    # 3️⃣ Save JSON to file
    # -----------------------------
    saved_path = None

    if save_json:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        os.makedirs(output_dir, exist_ok=True)

        saved_path = os.path.join(
            output_dir,
            f"{output_prefix}_{timestamp}.json"
        )

        with open(saved_path, "w", encoding="utf-8") as f:
            json.dump(analysis_result, f, indent=2)

        print(f"💾 Full analysis result saved to: {saved_path}")

    return saved_path


In [3]:
def print_normalized_table(json_path: str, normalized_suffix: str = "_normalized"):
    data = json.loads(open(json_path, "r", encoding="utf-8").read())
    content = data["result"]["contents"][0]
    fields = content.get("fields", {})

    kv = extract_kv_table(fields, suffix=normalized_suffix)

    print("\nNormalized fields:")
    print("-" * 110)
    for k, meta in kv.items():
        conf = meta.get("confidence")
        conf_str = f"{conf*100:.1f}%" if isinstance(conf, (int, float)) else "-"
        print(f"{k:40s} | {conf_str:>7s} | {meta.get('value','')}")


In [4]:
import json
import re
from collections import defaultdict
from typing import Any, Dict, List, Tuple, Optional

import matplotlib.pyplot as plt
import matplotlib.patches as patches

# -----------------------------
# Confidence → color
# -----------------------------
def confidence_color(conf: float | None) -> str:
    if conf is None:
        return "blue"
    if conf >= 0.90:
        return "green"
    elif conf >= 0.70:
        return "yellow"
    else:
        return "red"


# -----------------------------
# Parse all D(...) quads
# -----------------------------
def parse_all_source_quads(source: str) -> List[Tuple[int, List[Tuple[float, float]]]]:
    if not isinstance(source, str):
        return []
    quads = []
    for inner in re.findall(r"D\(([^)]+)\)", source):
        parts = [p.strip() for p in inner.split(",")]
        if len(parts) != 1 + 8:
            continue
        page_num = int(float(parts[0]))
        coords = list(map(float, parts[1:]))
        pts = list(zip(coords[0::2], coords[1::2]))
        if len(pts) == 4:
            quads.append((page_num, pts))
    return quads


def quad_to_pixels(pts, page_w, page_h, img_w, img_h):
    return [(x / page_w * img_w, y / page_h * img_h) for x, y in pts]


# -----------------------------
# Extract anchored items for ONE node
# -----------------------------
def extract_anchored_items(node: Any, path: str) -> List[Dict[str, Any]]:
    items = []
    if not isinstance(node, dict):
        return items

    text = node.get("valueString") or node.get("content") or node.get("value") or ""

    # top-level source
    if "source" in node and isinstance(node["source"], str):
        items.append({
            "path": path,
            "text": text,
            "confidence": node.get("confidence"),
            "source": node["source"]
        })

    # span-level sources
    spans = node.get("spans")
    if isinstance(spans, list):
        for i, sp in enumerate(spans):
            if isinstance(sp, dict) and isinstance(sp.get("source"), str):
                sp_text = sp.get("content") or text
                items.append({
                    "path": f"{path}.spans[{i}]",
                    "text": sp_text,
                    "confidence": sp.get("confidence", node.get("confidence")),
                    "source": sp["source"]
                })
    return items


# -----------------------------
# NEW: Only pull anchored items for *_raw fields (top-level fields only)
# -----------------------------
def extract_sources_from_raw_fields(fields: Dict[str, Any], raw_suffix: str = "_raw") -> List[Dict[str, Any]]:
    """
    Expects content["fields"] dict from ACU result.
    Only extracts anchored sources for top-level keys that end with _raw.
    """
    items: List[Dict[str, Any]] = []
    if not isinstance(fields, dict):
        return items

    for k, v in fields.items():
        if not k.endswith(raw_suffix):
            continue
        # ACU fields are typically objects like {"type":..., "valueString":..., "source":..., "spans":[...]}
        items.extend(extract_anchored_items(v, f"fields.{k}"))

    return items


# -----------------------------
# NEW: Build a clean key/value dict for normalized fields
# -----------------------------
def extract_kv_table(fields: Dict[str, Any], suffix: str) -> Dict[str, Dict[str, Any]]:
    """
    Returns:
      {
        "FieldName_normalized": {"value": "...", "confidence": 0.93},
        ...
      }
    """
    out: Dict[str, Dict[str, Any]] = {}
    if not isinstance(fields, dict):
        return out

    for k, v in fields.items():
        if not k.endswith(suffix):
            continue
        if not isinstance(v, dict):
            continue

        value = v.get("valueString") or v.get("content") or v.get("value") or ""
        out[k] = {
            "value": value,
            "confidence": v.get("confidence")
        }

    return out


In [5]:
def visualize_acu_raw_fields(json_path: str, pdf_images: List[Any], raw_suffix: str = "_raw", max_label_len: int = 50):
    data = json.loads(open(json_path, "r", encoding="utf-8").read())
    content = data["result"]["contents"][0]
    page_meta = {p["pageNumber"]: p for p in content.get("pages", [])}

    raw_items = extract_sources_from_raw_fields(content.get("fields", {}), raw_suffix=raw_suffix)

    by_page = defaultdict(list)
    for item in raw_items:
        for page_num, pts in parse_all_source_quads(item.get("source", "")):
            by_page[page_num].append((item, pts))

    for page_num in range(1, len(pdf_images) + 1):
        img = pdf_images[page_num - 1]
        page = page_meta.get(page_num)
        if not page:
            continue

        fig, ax = plt.subplots(1, 1, figsize=(15, 20))
        ax.imshow(img)
        ax.set_title(f"Page {page_num} - ACU Raw Fields (only '{raw_suffix}')", fontsize=16)

        items = by_page.get(page_num, [])
        if not items:
            ax.text(0.5, 0.02, "No raw boxes on this page", transform=ax.transAxes,
                    ha="center", va="bottom", fontsize=10, color="gray")
        else:
            for item, pts in items:
                px_pts = quad_to_pixels(
                    pts,
                    page["width"],
                    page["height"],
                    img.width,
                    img.height
                )

                conf = item.get("confidence")
                color = confidence_color(conf)

                poly = patches.Polygon(
                    px_pts,
                    closed=True,
                    linewidth=1.8,
                    edgecolor=color,
                    facecolor="none",
                    alpha=0.9
                )
                ax.add_patch(poly)

                x0 = min(p[0] for p in px_pts)
                y0 = min(p[1] for p in px_pts)

                # Label = field key (not the whole raw text, which gets noisy)
                # Example path: "fields.DocumentName_raw.spans[0]" -> want "DocumentName_raw"
                path = item.get("path", "")
                field_key = path.split(".")[1] if path.startswith("fields.") and "." in path else path

                label = field_key
                if conf is not None:
                    label = f"{label} ({conf*100:.1f}%)"

                ax.annotate(
                    label[:max_label_len] + ("..." if len(label) > max_label_len else ""),
                    (x0, max(0, y0 - 5)),
                    bbox=dict(
                        boxstyle="round,pad=0.3",
                        facecolor="white",
                        edgecolor=color,
                        alpha=0.95
                    ),
                    fontsize=8,
                    color="black",
                    ha="left",
                    va="bottom",
                )

        legend_elements = [
            patches.Patch(color="green", label="High Confidence (≥ 90%)"),
            patches.Patch(color="yellow", label="Medium Confidence (70–89%)"),
            patches.Patch(color="red", label="Low Confidence (< 70%)"),
        ]
        ax.legend(handles=legend_elements, loc="upper right", fontsize=9, framealpha=0.95)

        ax.set_xlim(0, img.width)
        ax.set_ylim(img.height, 0)
        ax.axis("off")
        plt.tight_layout()
        plt.show()


### Create Analyzer

In [6]:
import json

license_analyzer_id = "license_agreement_extraction_wrt_CUAD_v4_raw_normalized_singlepass"

# Tune these to control verbosity (big lever for speed + output tokens)
RAW_CHAR_CAP = 350          # keep raw extracts short for bbox anchoring
RAW_SENTENCE_CAP = 2        # additional guardrail for long clauses

def raw_desc(base: str) -> str:
    return (
        f"{base} "
        f"VERBATIM ONLY: copy exact text from the document (no paraphrase, no reformatting, no normalization). "
        f"Return the minimal span necessary to answer. "
        f"Hard limits: max {RAW_SENTENCE_CAP} sentences OR ~{RAW_CHAR_CAP} characters (whichever is smaller). "
        f"If the value is not present, return an empty string."
    )

license_agreement_analyzer = {
    "baseAnalyzerId": "prebuilt-document",
    "description": (
        "Single-pass analyzer for License Agreements that outputs BOTH raw (verbatim, source-anchored) "
        "and normalized (machine-friendly) fields. Use *_raw for bounding boxes; use *_normalized for DB/evaluation. "
        "Raw fields are intentionally capped to reduce latency and token usage."
    ),
    "config": {
        # FINAL RECOMMENDATION: keep one analyzer, keep returnDetails False
        "returnDetails": False,
        "enableOcr": True,   # set to False if PDFs are always digitally generated (copy/paste works)
        "enableLayout": True,  # keep True so sources/quads can be produced reliably
        "estimateFieldSourceAndConfidence": True  # needed for bbox (field-level source quads)
    },
    "fieldSchema": {
        "name": "LicenseAgreementFields_RawNormalized_SinglePass",
        "fields": {
            # 1) Document Name
            "DocumentName_raw": {
                "type": "string",
                "method": "generate",
                "description": raw_desc("Official title of the agreement.")
            },
            "DocumentName_normalized": {
                "type": "string",
                "method": "generate",
                "description": "Clean official title as a single line. Remove extra whitespace/line breaks."
            },

            # 2) Parties
            "Parties_raw": {
                "type": "string",
                "method": "generate",
                "description": raw_desc(
                    "Parties to the agreement as written (including defined terms / aliases in quotes or parentheses)."
                )
            },
            "Parties_normalized": {
                "type": "string",
                "method": "generate",
                "description": (
                    "Normalized parties list. Separate multiple parties with '; '. "
                    "Preserve aliases in parentheses if present. Remove extra whitespace."
                )
            },

            # 3) Agreement Date
            "AgreementDate_raw": {
                "type": "string",
                "method": "generate",
                "description": raw_desc("Execution/signing date text as written (do not convert format).")
            },
            "AgreementDate_normalized": {
                "type": "string",
                "method": "generate",
                "description": "Agreement execution date in mm/dd/yyyy. If not present, return empty string."
            },

            # 4) Effective Date
            "EffectiveDate_raw": {
                "type": "string",
                "method": "generate",
                "description": raw_desc("Effective date text as written (may include label like 'Effective Date').")
            },
            "EffectiveDate_normalized": {
                "type": "string",
                "method": "generate",
                "description": (
                    "Effective date in mm/dd/yyyy. If only relative wording exists and no explicit date is stated, "
                    "return empty string."
                )
            },

            # 5) Expiration Date / Term End
            "ExpirationDate_raw": {
                "type": "string",
                "method": "generate",
                "description": raw_desc(
                    "Initial term expiration/end date text as written, or wording indicating perpetual/evergreen."
                )
            },
            "ExpirationDate_normalized": {
                "type": "string",
                "method": "generate",
                "description": (
                    "Return expiration date in mm/dd/yyyy if explicitly stated; else return 'Perpetual' if perpetual/evergreen; "
                    "else empty string."
                )
            },

            # 6) Renewal Term
            "RenewalTerm_raw": {
                "type": "string",
                "method": "generate",
                "description": raw_desc("Renewal language as written (auto-renew, successive terms, conditions).")
            },
            "RenewalTerm_normalized": {
                "type": "string",
                "method": "generate",
                "description": (
                    "Concise structured phrase, e.g. 'successive 1 year', '2 years', 'month-to-month', "
                    "'perpetual', or 'no renewal'. Not a paragraph."
                )
            },

            # 7) Notice Period to Terminate Renewal
            "NoticeToTerminateRenewal_raw": {
                "type": "string",
                "method": "generate",
                "description": raw_desc(
                    "Notice period wording required to prevent renewal (keep units/format as written)."
                )
            },
            "NoticeToTerminateRenewal_normalized": {
                "type": "string",
                "method": "generate",
                "description": (
                    "Concise notice period like '60 days' or '30 days'. Use the notice that applies to stopping renewal."
                )
            },

            # 8) Governing Law
            "GoverningLaw_raw": {
                "type": "string",
                "method": "generate",
                "description": raw_desc("Governing law clause fragment containing the jurisdiction as written.")
            },
            "GoverningLaw_normalized": {
                "type": "string",
                "method": "generate",
                "description": "Jurisdiction only (state/country/province). Example: 'California'."
            },

            # 9) License Grant
            "LicenseGrant_raw": {
                "type": "string",
                "method": "generate",
                "description": raw_desc(
                    "License grant clause text as written (scope/rights grant). Return minimal span; avoid long paragraphs."
                )
            },
            "LicenseGrant_normalized": {
                "type": "string",
                "method": "generate",
                "description": (
                    "Structured summary as key=value pairs, e.g. "
                    "'type=trademark; exclusivity=non-exclusive; territory=worldwide; sublicensing=yes; transferable=no'."
                )
            },

            # 10) Exclusivity
            "Exclusivity_raw": {
                "type": "string",
                "method": "generate",
                "description": raw_desc("Text indicating exclusivity/non-exclusivity as written.")
            },
            "Exclusivity_normalized": {
                "type": "string",
                "method": "generate",
                "description": "Return 'Yes' if exclusive else 'No'. If unclear, empty string."
            },

            # 11) Termination for Convenience
            "TerminationForConvenience_raw": {
                "type": "string",
                "method": "generate",
                "description": raw_desc("Text indicating termination without cause (termination for convenience) as written.")
            },
            "TerminationForConvenience_normalized": {
                "type": "string",
                "method": "generate",
                "description": "Return 'Yes' if either party may terminate without cause else 'No'. If unclear, empty string."
            },
        }
    },
    "models": {
        "completion": "gpt-4.1-mini"
    }
}

print(json.dumps(license_agreement_analyzer, indent=2))

resp = client.begin_create_analyzer(
    analyzer_id=license_analyzer_id,
    analyzer_template=license_agreement_analyzer,
)

print("⏳ Waiting for analyzer creation to complete...")
client.poll_result(resp)
print(f"✅ Analyzer '{license_analyzer_id}' created successfully!")


{
  "baseAnalyzerId": "prebuilt-document",
  "description": "Single-pass analyzer for License Agreements that outputs BOTH raw (verbatim, source-anchored) and normalized (machine-friendly) fields. Use *_raw for bounding boxes; use *_normalized for DB/evaluation. Raw fields are intentionally capped to reduce latency and token usage.",
  "config": {
    "returnDetails": false,
    "enableOcr": true,
    "enableLayout": true,
    "estimateFieldSourceAndConfidence": true
  },
  "fieldSchema": {
    "name": "LicenseAgreementFields_RawNormalized_SinglePass",
    "fields": {
      "DocumentName_raw": {
        "type": "string",
        "method": "generate",
        "description": "Official title of the agreement. VERBATIM ONLY: copy exact text from the document (no paraphrase, no reformatting, no normalization). Return the minimal span necessary to answer. Hard limits: max 2 sentences OR ~350 characters (whichever is smaller). If the value is not present, return an empty string."
      },
   

INFO:content_understanding_client:Analyzer license_agreement_extraction_wrt_CUAD_v4_raw_normalized_singlepass create request accepted.


⏳ Waiting for analyzer creation to complete...


INFO:content_understanding_client:Request result is ready after 0.00 seconds.


✅ Analyzer 'license_agreement_extraction_wrt_CUAD_v4_raw_normalized_singlepass' created successfully!


### Analyze document

In [7]:
sample_file_path = 'data/AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf'
license_analyzer_id = "license_agreement_extraction_wrt_CUAD_v3_raw_normalized_singlepass"
# Begin document analysis operation
print(f"🔍 Starting document analysis with analyzer '{license_analyzer_id}'...")
analysis_response = client.begin_analyze_binary(
    analyzer_id=license_analyzer_id,
    file_location=sample_file_path,
)

# Wait for analysis completion
print(f"⏳ Waiting for document analysis to complete...")
analysis_result = client.poll_result(analysis_response)
print(f"✅ Document analysis completed successfully!")

🔍 Starting document analysis with analyzer 'license_agreement_extraction_wrt_CUAD_v3_raw_normalized_singlepass'...


INFO:content_understanding_client:Analyzing binary file data/AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf with analyzer: license_agreement_extraction_wrt_CUAD_v3_raw_normalized_singlepass


⏳ Waiting for document analysis to complete...


INFO:content_understanding_client:Request c7face6c-907a-41f9-91aa-2718fe8b304f in progress ...
INFO:content_understanding_client:Request c7face6c-907a-41f9-91aa-2718fe8b304f in progress ...
INFO:content_understanding_client:Request c7face6c-907a-41f9-91aa-2718fe8b304f in progress ...
INFO:content_understanding_client:Request c7face6c-907a-41f9-91aa-2718fe8b304f in progress ...
INFO:content_understanding_client:Request c7face6c-907a-41f9-91aa-2718fe8b304f in progress ...
INFO:content_understanding_client:Request c7face6c-907a-41f9-91aa-2718fe8b304f in progress ...
INFO:content_understanding_client:Request c7face6c-907a-41f9-91aa-2718fe8b304f in progress ...
INFO:content_understanding_client:Request c7face6c-907a-41f9-91aa-2718fe8b304f in progress ...
INFO:content_understanding_client:Request c7face6c-907a-41f9-91aa-2718fe8b304f in progress ...
INFO:content_understanding_client:Request c7face6c-907a-41f9-91aa-2718fe8b304f in progress ...
INFO:content_understanding_client:Request c7face6c

✅ Document analysis completed successfully!


### Save Analysis as JSON

In [8]:
raw_file = process_and_save_analysis(
    analysis_result,
    output_prefix="license_analysis_combined"
)


📊 Extracted Fields:
--------------------------------------------------------------------------------
DocumentName_raw: JOINT CONTENT LICENSE AGREEMENT

DocumentName_normalized: JOINT CONTENT LICENSE AGREEMENT

Parties_raw: WPT Enterprises, Inc., a Delaware corporation, with offices located at 1920 Main Street, Suite 1150, Irvine, CA 92614 ("WPT"), and ZYNGA INC., a Delaware corporation with offices located at 699 8th Street, San Francisco CA, 94103 ("Zynga US") and ZYNGA GAME IRELAND LIMITED, a limited company organized under the laws of Ireland, resident in Ireland and having its registered office located at The Oval, Building One, Third Floor 160 Shelbourne Road Ballsbridge 4 Co. Dublin Ireland ("Zynga Ireland," and together with Zynga US and their respective Affiliates, "Zynga").

Parties_normalized: WPT Enterprises, Inc. ("WPT"); ZYNGA INC. ("Zynga US"); ZYNGA GAME IRELAND LIMITED ("Zynga Ireland")

AgreementDate_raw: dated February 1, 2018

AgreementDate_normalized: 02/01/2018

E

### Visualize PDF

In [ ]:
from pdf2image import convert_from_path

pdf_images = convert_from_path(
    "data/AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf",
    poppler_path=r"C:\Users\deril\poppler\Library\bin"
)

json_path = r"C:\Users\deril\OneDrive\Desktop\Deril\Development\Azure\Intelligent-Document-Processing-Production-Ready\test_output\license_analysis_combined_20260211_134140.json"

# 1) Draw boxes only for *_raw
visualize_acu_raw_fields(json_path, pdf_images)

### Print Normalized fields

In [ ]:
print_normalized_table(json_path)